# 02 · Silver — Cleansing and conformed staging

Reads from `bronze.*` (active snapshot only) and writes `silver.stg_*` tables. Operates only on `engine_dwh` (VPN OFF).

## 1. Imports and engine

In [1]:
import pandas as pd
from datetime import datetime, timezone
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from dotenv import load_dotenv
import os

load_dotenv(override=True)

SUPABASE_URL  = os.getenv("SUPABASE_URL")
SUPABASE_PORT = int(os.getenv("SUPABASE_PORT", 5432))
SUPABASE_DB   = os.getenv("SUPABASE_DB")
SUPABASE_USER = os.getenv("SUPABASE_USER")
SUPABASE_PASSWORD = os.getenv("SUPABASE_PASSWORD")

def make_engine(host, port, db, user, password, sslmode=None):
    return create_engine(URL.create("postgresql+psycopg2", username=user, password=password, host=host, port=port, database=db, query={"sslmode": sslmode} if sslmode else None))

if not all([SUPABASE_URL, SUPABASE_DB, SUPABASE_USER, SUPABASE_PASSWORD]):
    raise ValueError("SUPABASE env vars missing")
engine_dwh = make_engine(SUPABASE_URL, SUPABASE_PORT, SUPABASE_DB, SUPABASE_USER, SUPABASE_PASSWORD, sslmode="require")
run_at = datetime.now(timezone.utc).replace(tzinfo=None)
print("✓ engine_dwh ready.")


✓ engine_dwh ready.


## 2. DDL — silver staging tables (drop and recreate)

Silver is non-historized — every run rebuilds the staging tables. Idempotent.

In [2]:
DDL_SILVER = {
    "stg_employees": """
        CREATE TABLE IF NOT EXISTS silver.stg_employees (
            personid INT NOT NULL,
            fullname TEXT,
            issalesperson INT,
            _loaded_at TIMESTAMP NOT NULL,
            _source_snapshot INT NOT NULL
        );
    """,
    "stg_customers": """
        CREATE TABLE IF NOT EXISTS silver.stg_customers (
            customerid INT NOT NULL,
            customername TEXT,
            category TEXT,
            deliverycityid INT,
            _loaded_at TIMESTAMP NOT NULL,
            _source_snapshot INT NOT NULL
        );
    """,
    "stg_locations": """
        CREATE TABLE IF NOT EXISTS silver.stg_locations (
            locationid INT NOT NULL,
            city TEXT,
            state TEXT,
            country TEXT,
            salesterritory TEXT,
            _loaded_at TIMESTAMP NOT NULL,
            _source_snapshot INT NOT NULL
        );
    """,
    "stg_products": """
        CREATE TABLE IF NOT EXISTS silver.stg_products (
            stockitemid INT NOT NULL,
            stockitemname TEXT,
            brand TEXT,
            _loaded_at TIMESTAMP NOT NULL,
            _source_snapshot INT NOT NULL
        );
    """,
    "stg_deliverymethods": """
        CREATE TABLE IF NOT EXISTS silver.stg_deliverymethods (
            deliverymethodid INT NOT NULL,
            deliverymethodname TEXT,
            _loaded_at TIMESTAMP NOT NULL,
            _source_snapshot INT NOT NULL
        );
    """,
    "stg_paymentmethods": """
        CREATE TABLE IF NOT EXISTS silver.stg_paymentmethods (
            paymentmethodid INT NOT NULL,
            paymentmethodname TEXT,
            _loaded_at TIMESTAMP NOT NULL,
            _source_snapshot INT NOT NULL
        );
    """,
    "stg_transactiontypes": """
        CREATE TABLE IF NOT EXISTS silver.stg_transactiontypes (
            transactiontypeid INT NOT NULL,
            transactiontypename TEXT,
            _loaded_at TIMESTAMP NOT NULL,
            _source_snapshot INT NOT NULL
        );
    """,
    "stg_fact_sales": """
        CREATE TABLE IF NOT EXISTS silver.stg_fact_sales (
            saleskey INT NOT NULL,
            invoiceid INT NOT NULL,
            invoicedate DATE,
            salespersonpersonid INT,
            accountspersonid INT,
            customerid INT,
            stockitemid INT,
            deliverycityid INT,
            deliverymethodid INT,
            quantity INT,
            unitprice NUMERIC,
            taxamount NUMERIC,
            extendedprice NUMERIC,
            lineprofit NUMERIC,
            _loaded_at TIMESTAMP NOT NULL
        );
    """,
    "stg_fact_invoices": """
        CREATE TABLE IF NOT EXISTS silver.stg_fact_invoices (
            invoiceid INT NOT NULL,
            invoicedate DATE,
            salespersonpersonid INT,
            accountspersonid INT,
            customerid INT,
            deliverycityid INT,
            deliverymethodid INT,
            invoiceamount NUMERIC,
            paymentdelay_days INT,
            outstandingbalance NUMERIC,
            _loaded_at TIMESTAMP NOT NULL
        );
    """,
}

with engine_dwh.begin() as conn:
    for name, ddl in DDL_SILVER.items():
        conn.execute(text(f"DROP TABLE IF EXISTS silver.{name} CASCADE"))
        conn.execute(text(ddl))
print("✓ Silver staging tables (re)created.")


✓ Silver staging tables (re)created.


## 3. Helper functions

In [3]:
def read_active_snapshot(table_name):
    sql = f"SELECT * FROM bronze.{table_name} WHERE _snapshot_id = (SELECT MAX(_snapshot_id) FROM bronze.{table_name}) AND _change_op != 'DELETE'"
    with engine_dwh.connect() as conn:
        return pd.read_sql(text(sql), conn)

def clean_str(s):
    """Strip whitespace and convert empty strings to NaN. Tolerates non-string inputs."""
    return s.astype("string").str.strip().replace({"": pd.NA})


## 4. stg_employees

In [4]:
print("Processing stg_employees...")
df_people = read_active_snapshot("people")
if not df_people.empty:
    snap_id = int(df_people["_snapshot_id"].max())
    out = pd.DataFrame({
        "personid": df_people["personid"],
        "fullname": clean_str(df_people["fullname"]),
        "issalesperson": df_people["issalesperson"],
        "_loaded_at": run_at,
        "_source_snapshot": snap_id,
    })
    with engine_dwh.begin() as conn:
        out.to_sql("stg_employees", conn, schema="silver", if_exists="append", index=False)
    print(f"  stg_employees: {len(out)} records")
else:
    print("  stg_employees: no source data")


Processing stg_employees...
  stg_employees: 1111 records


## 5. stg_customers

In [5]:
print("Processing stg_customers...")
df_cust = read_active_snapshot("customers")
df_cat = read_active_snapshot("customercategories")
if not df_cust.empty:
    snap_id = int(df_cust["_snapshot_id"].max())
    df = df_cust.merge(df_cat[["customercategoryid", "customercategoryname"]], on="customercategoryid", how="left")
    out = pd.DataFrame({
        "customerid": df["customerid"],
        "customername": clean_str(df["customername"]),
        "category": clean_str(df["customercategoryname"]),
        "deliverycityid": df["deliverycityid"],
        "_loaded_at": run_at,
        "_source_snapshot": snap_id,
    })
    with engine_dwh.begin() as conn:
        out.to_sql("stg_customers", conn, schema="silver", if_exists="append", index=False)
    print(f"  stg_customers: {len(out)} records")
else:
    print("  stg_customers: no source data")


Processing stg_customers...
  stg_customers: 663 records


## 6. stg_locations

In [6]:
print("Processing stg_locations...")
df_cities = read_active_snapshot("cities")
df_states = read_active_snapshot("stateprovinces")
df_countries = read_active_snapshot("countries")
if not df_cities.empty:
    snap_id = int(df_cities["_snapshot_id"].max())
    df = (df_cities.merge(df_states[["stateprovinceid", "stateprovincename", "countryid", "salesterritory"]], on="stateprovinceid", how="left")
                    .merge(df_countries[["countryid", "ountryname"]], on="countryid", how="left"))
    out = pd.DataFrame({
        "locationid": df["cityid"],
        "city": clean_str(df["cityname"]),
        "state": clean_str(df["stateprovincename"]),
        "country": clean_str(df["ountryname"]),
        "salesterritory": clean_str(df["salesterritory"]),
        "_loaded_at": run_at,
        "_source_snapshot": snap_id,
    })
    with engine_dwh.begin() as conn:
        out.to_sql("stg_locations", conn, schema="silver", if_exists="append", index=False)
    print(f"  stg_locations: {len(out)} records")
else:
    print("  stg_locations: no source data")

Processing stg_locations...
  stg_locations: 37940 records


## 7. stg_products

In [7]:
print("Processing stg_products...")
df_prod = read_active_snapshot("stockitems")
if not df_prod.empty:
    out = pd.DataFrame({
        "stockitemid": df_prod["stockitemid"],
        "stockitemname": clean_str(df_prod["stockitemname"]),
        "brand": clean_str(df_prod["brand"]),
        "_loaded_at": run_at,
        "_source_snapshot": int(df_prod["_snapshot_id"].max()),
    })
    with engine_dwh.begin() as conn:
        out.to_sql("stg_products", conn, schema="silver", if_exists="append", index=False)
    print(f"  stg_products: {len(out)} records")
else:
    print("  stg_products: no source data")


Processing stg_products...
  stg_products: 227 records


## 8. stg_deliverymethods / stg_paymentmethods / stg_transactiontypes

In [8]:
for src_table, stg_table, key_col, name_col in [
    ("deliverymethods", "stg_deliverymethods", "deliverymethodid", "deliverymethodname"),
    ("paymentmethods", "stg_paymentmethods", "paymentmethodid", "paymentmethodname"),
    ("transactiontypes", "stg_transactiontypes", "transactiontypeid", "transactiontypename"),
]:
    print(f"Processing {stg_table}...")
    try:
        df = read_active_snapshot(src_table)
    except Exception as e:
        print(f"  ✗ {stg_table}: {e}")
        continue
    if df.empty:
        print(f"  {stg_table}: no source data")
        continue
    out = pd.DataFrame({
        key_col: df[key_col],
        name_col: clean_str(df[name_col]),
        "_loaded_at": run_at,
        "_source_snapshot": int(df["_snapshot_id"].max()),
    })
    with engine_dwh.begin() as conn:
        out.to_sql(stg_table, conn, schema="silver", if_exists="append", index=False)
    print(f"  {stg_table}: {len(out)} records")


Processing stg_deliverymethods...
  stg_deliverymethods: 10 records
Processing stg_paymentmethods...
  stg_paymentmethods: 4 records
Processing stg_transactiontypes...
  stg_transactiontypes: 13 records


## 9. stg_fact_sales and stg_fact_invoices

Joins:
- `bronze.invoicelines` ⨝ `bronze.invoices` (header info: salesperson, accounts person, customer, delivery method)
- ⨝ `bronze.customers` (delivery city → location)
- For invoice header: aggregate invoicelines for `invoiceamount`; left-join `bronze.customertransactions` for `paymentdelay_days` and `outstandingbalance`.

In [9]:
print("Processing stg_fact_sales & stg_fact_invoices...")
with engine_dwh.connect() as conn:
    df_inv = pd.read_sql(text("SELECT * FROM bronze.invoices"), conn)
    df_lines = pd.read_sql(text("SELECT * FROM bronze.invoicelines"), conn)
df_cust = read_active_snapshot("customers")
try:
    df_trans = read_active_snapshot("customertransactions")
except Exception:
    df_trans = pd.DataFrame()

if df_inv.empty or df_lines.empty:
    print("  No invoices/invoicelines in bronze. Skipping facts.")
else:
    inv_cols = ["invoiceid", "invoicedate", "salespersonpersonid", "accountspersonid", "customerid"]
    if "deliverymethodid" in df_inv.columns:
        inv_cols.append("deliverymethodid")
    df_sales = (df_lines.merge(df_inv[inv_cols], on="invoiceid", how="left")
                       .merge(df_cust[["customerid", "deliverycityid"]], on="customerid", how="left"))

    stg_fact_sales = pd.DataFrame({
        "saleskey": df_sales["invoicelineid"],
        "invoiceid": df_sales["invoiceid"],
        "invoicedate": pd.to_datetime(df_sales["invoicedate"]),
        "salespersonpersonid": df_sales["salespersonpersonid"],
        "accountspersonid": df_sales["accountspersonid"],
        "customerid": df_sales["customerid"],
        "stockitemid": df_sales["stockitemid"],
        "deliverycityid": df_sales["deliverycityid"],
        "deliverymethodid": df_sales["deliverymethodid"] if "deliverymethodid" in df_sales.columns else None,
        "quantity": df_sales["quantity"],
        "unitprice": df_sales["unitprice"],
        "taxamount": df_sales["taxamount"],
        "extendedprice": df_sales["extendedprice"],
        "lineprofit": df_sales["lineprofit"],
        "_loaded_at": run_at,
    })

    df_header = df_inv.merge(df_cust[["customerid", "deliverycityid"]], on="customerid", how="left")
    invoice_amounts = df_lines.groupby("invoiceid")["extendedprice"].sum().reset_index().rename(columns={"extendedprice": "invoiceamount"})
    df_header = df_header.merge(invoice_amounts, on="invoiceid", how="left")

    if not df_trans.empty:
        trans = df_trans.groupby("invoiceid").agg({
            "outstandingbalance": "sum",
            "finalizationdate": "max",
            "transactiondate": "min",
        }).reset_index()
        df_header = df_header.merge(trans, on="invoiceid", how="left")
        df_header["paymentdelay_days"] = (pd.to_datetime(df_header["finalizationdate"]) - pd.to_datetime(df_header["invoicedate"])).dt.days
        df_header["paymentdelay_days"] = df_header["paymentdelay_days"].fillna(0).astype(int)
        df_header["outstandingbalance"] = df_header["outstandingbalance"].fillna(0)
    else:
        df_header["paymentdelay_days"] = 0
        df_header["outstandingbalance"] = 0

    stg_fact_invoices = pd.DataFrame({
        "invoiceid": df_header["invoiceid"],
        "invoicedate": pd.to_datetime(df_header["invoicedate"]),
        "salespersonpersonid": df_header["salespersonpersonid"],
        "accountspersonid": df_header["accountspersonid"],
        "customerid": df_header["customerid"],
        "deliverycityid": df_header["deliverycityid"],
        "deliverymethodid": df_header["deliverymethodid"] if "deliverymethodid" in df_header.columns else None,
        "invoiceamount": df_header["invoiceamount"],
        "paymentdelay_days": df_header["paymentdelay_days"],
        "outstandingbalance": df_header["outstandingbalance"],
        "_loaded_at": run_at,
    })

    with engine_dwh.begin() as conn:
        stg_fact_sales.to_sql("stg_fact_sales", conn, schema="silver", if_exists="append", index=False)
        stg_fact_invoices.to_sql("stg_fact_invoices", conn, schema="silver", if_exists="append", index=False)
    print(f"  stg_fact_sales: {len(stg_fact_sales)} records")
    print(f"  stg_fact_invoices: {len(stg_fact_invoices)} records")


Processing stg_fact_sales & stg_fact_invoices...
  stg_fact_sales: 228265 records
  stg_fact_invoices: 70510 records
